# KMA First Pass

In [1]:
import os
import json
import dspy
import numpy as np
import ast
from pydantic import BaseModel, Field
from typing import List
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

cwd = os.getcwd()

/home/marwas/keymetrics/kmavenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#vllm command

# CUDA_VISIBLE_DEVICES=0 vllm serve casperhansen/deepseek-r1-distill-qwen-32b-awq --max-model-len 32768 --gpu-memory-utilization 0.9 --quantization awq_marlin --port 8000
### if above fails try:#### CUDA_VISIBLE_DEVICES=0 vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-14B --max-model-len 32768 --gpu-memory-utilization 0.9 --quantization awq_marlin --port 8000

In [2]:
vllm_model = dspy.LM(
    api_base="http://localhost:8000/v1",
    api_key="local",
    model="openai/casperhansen/deepseek-r1-distill-qwen-32b-awq",
    max_tokens=4096
)
dspy.configure(lm=vllm_model)

In [26]:
class ExtMetric(BaseModel):
    # using pydantic to ensure our outputs are the right data types
    extracted_text: str = Field(description="The word or phrase from the text")
    metric_level: str = Field(description="Action, Outcome, or Unsure")
    metric_class: str = Field(description="Area, Emissions, Spending, etc.")

class PolProc(dspy.Signature):
    """Extract peatland-related policy metrics that could be examined to evlauate a policy's delivery or impact. 
    Ignore all non-peatland-related metrics."""
    text: str = dspy.InputField(desc="Policy document text to analyze.")
    output: List[ExtMetric] = dspy.OutputField(desc="""For each relevant metric, return the relevant text along with the metric's level and class classification using the following lists as labels.
    Levels:
    - Action (actions that will be taken as part of the policy's implementation)
    - Outcome (intended or expected consequences of the policy's successful and effective implementation, sometimes an objective or goal) 
    - Unsure (unclear or other)
    Classes: 
    - Area (e.g. km2, ha, %)
    - Emissions (e.g. tCO2e, % reduction)
    - Site Status (e.g. site designation, protected status)
    - Spending (e.g. €, budget)
    - Policy Action (e.g. regulations, policies, laws, schemes, programmes, infrastructure like paludiculture market)
    - Knowledge Resource (e.g. reports, guides, analyses, trainings, information campaigns)
    - Practical Resource (e.g. supplies, equipment, restoration materials)
    - Environment Quality (e.g. water level, biodiversity, vegetation types, indicators of soil or ecosystem health)
    - Miscellaneous (e.g. other)""")

## One Chunk of Policy Text at a Time
quick n dirty ai

This would be implemented by running the following on every paragraph in every policy document. Obviously not ideal computationally, but it demonstrates a proof of concept.

In [15]:
# generated a chunk of policy text with a high density of relevant and irrelevant metrics of different classes
gen_test_text = """
    Peatland Restoration and Climate Action Policy 2026
    1. Land Management and Conservation Targets
    The Department aims to enhance the resilience of national carbon sinks through active intervention. 
    By 2030, the government commits to the rewetting of 50,000 hectares of degraded raised bogs. 
    This target focuses on the SAC-designated North-Western Complex, which has been prioritized for immediate hydrological restoration. 
    Conversely, the urbanization of suburban zones has increased by 12% in the last decade, a trend that must be decoupled from environmental planning.
    2. Emissions Reductions and Air Quality
    A primary objective of this framework is the mitigation of greenhouse gas release. 
    We anticipate a total reduction of 4.2 million tCO2e by 2040 specifically from peatland re-vegetation. 
    This is distinct from our national goal to reduce passenger vehicle emissions by 15%, which is handled under the Transport Directive. 
    Furthermore, the Peatland Carbon Assessment Report will be published annually to track progress.
    3. Financial Allocation and Scheme Development
    To support rural communities, the Minister has authorized the Peatland Agri-Environmental Scheme (PAES) to provide direct support to landowners. 
    A dedicated budget of €120 million has been ring-fenced for this purpose. 
    In comparison, the Department of Education has noted that student enrollment has reached 600,000 pupils, requiring a separate infrastructure budget not covered under this environmental mandate.
    4. Technical Specifications and Restoration Depth
    Restoration success is measured by the stability of the water table. 
    We require the installation of 450 plastic piling dams per site to ensure water levels remain within 10cm of the surface. 
    During the pilot phase, the average temperature of the site was recorded at 18°C, which, while noted by researchers, is not a metric for policy delivery or outcome. 
"""

In [16]:
test_proc = dspy.ChainOfThought(PolProc)
pred = test_proc(text=gen_test_text)
# about 1 minute

In [7]:
for entry in pred.output:
    print(f"{entry.extracted_text} | {entry.metric_level} | {entry.metric_class}")

50,000 hectares | Outcome | Area
SAC-designated North-Western Complex | Outcome | Site Status
4.2 million tCO2e | Outcome | Emissions
Peatland Carbon Assessment Report | Outcome | Knowledge Resource
€120 million | Action | Spending
450 plastic piling dams | Action | Practical Resource
10cm of the surface | Outcome | Environment Quality
18°C | Unsure | Environment Quality


### Evaluation
#### Correct
- 50,000 hectares | Outcome | Area
- _ignored_ "urbanization of suburban zones has increased by 12%"
- 4.2 million tCO2e | Outcome | Emissions
- _ignored_ "vehicle emissions by 15%"
- Peatland Carbon Assessment Report | Action | Knowledge Resource
- €120 million | Action | Spending
- _ignored_ "600,000 pupils"
- 450 plastic piling dams | Action | Practical Resource
- 10cm of the surface | Outcome | Environment Quality

#### Partially-Correct
- SAC-designated North-Western Complex | {Outcome} but should be {Unsure}? | Site Status
- Peatland Carbon Assessment Report | {Outcome} but should be {Action}? | Knowledge Resource
> Maybe Level doesn't make sense for binary metric classes like Site Status and Knowledge Resource?

#### Incorrect
- _missed_ Peatland Agri-Environmental Scheme (PAES) | Action | Policy Action
- _included_ 18°C | Unsure | Environment Quality

## One Policy Document at a Time

#### Qdrant

In [ ]:
#sudo docker run -p 6333:6333 -p 6334:6334 -v $(pwd)/qdrant_storage:/qdrant/storage:z qdrant/qdrant

In [3]:
dspy.settings.configure(lm=vllm_model)

In [4]:
client = QdrantClient("http://localhost:6333")
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device="cuda:2")

/home/marwas/keymetrics/kmavenv/lib/python3.10/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.14.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9681.78it/s]


need to work more on the following structure
- i think the qdrant retriever makes more sense using thresholding than k count
- how to structure the RAG signature to reflect the original COT signature better
    - do we even want a question field? i feel like no
- how to return the chunk ids referenced??

In [ ]:
def qdrant_retriever(query, k=100):
    query_vector = encoder.encode(query).tolist()
    response = client.query_points(
        collection_name="irish_cap", # Yasar's is nlp_docs
        query=query_vector,
        limit=k
    )
    return [hit.payload['text'] for hit in response.points]

In [ ]:
class PolProcRAGSig(dspy.Signature):
    """Extract peatland-related policy metrics into a list of tuples: ([text], [level], [class])."""
    context = dspy.InputField(desc="Relevant policy document chunks.")
    question = dspy.InputField(desc="What metrics should we look for?")
    output = dspy.OutputField(desc="List of metric tuples found.")

class PolProcRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(PolProcRAGSig)
    def forward(self, question):
        context_chunks = qdrant_retriever(question)
        context_text = "\n\n".join(context_chunks)
        prediction = self.generate_answer(context=context_text, question=question)
        return dspy.Prediction(
            metrics=prediction.output,
            reasoning=prediction.reasoning,
            sources=context_chunks
        )

In [ ]:
rag_system = PolProcRAG()
query = "What are the specific area and spending targets affecting peatlands?"
result = rag_system(query)
# k=100; 1 min

In [40]:
print(f"REASONING:\n{result.reasoning}\n")
print(f"EXTRACTED METRICS:\n{result.metrics}")
print(f"SOURCES:\n{result.sources}")

REASONING:
The context provides several metrics related to peatlands, particularly in terms of water table management and area targets. The AECM (Agri-Environment Climate Measure) strategy mentions specific targets for peat-based soils and wetlands, aiming to manage water tables and protect habitats. The text also references specific percentages and areas related to Natura 2000 sites and the coverage of agricultural land under the AECM framework.

The key metrics extracted include:
1. A target of water table management for 40,000 hectares of drained peatlands.
2. Rewetting of blanket bog areas under agricultural use.
3. Aiming for 10% of agricultural land (469,149 hectares) under high-diversity landscape features through AECM cooperation projects.

These metrics are tied to broader environmental goals, such as biodiversity preservation and carbon storage, as outlined in the policy document.

EXTRACTED METRICS:
```plaintext
- ('40,000 hectares of drained, agricultural, managed, carbon-r

### RAG Reconfigured